## Prepare Data
### Import all three datasets:
    Average Salary and Threshold
    Advertiser Sign-Ups
    Existing Agent Staffing
<b> Before building any tools, we will perform the following data cleanse:  
1. Eligible Ads:    
    - If Annual Budget > Threshold, then Eligible.    
2. Remove all Feb 29, 2025
>> </b>

In [4]:
#%pip install pandas openpyxl

In [59]:
import pandas as pd

url = "https://raw.githubusercontent.com/aschan12/DADS5250Hackathon/main/DADS5250_Hackathon_Google_Data.xlsx"

# 2. Read directly into a DataFrame
df_salary_threshold = pd.read_excel(url, sheet_name="Agent Salary and Threshold", engine="openpyxl")
df_adv_sign_ups = pd.read_excel(url, sheet_name="Advertiser Sign-ups", engine="openpyxl")
df_exist_staff = pd.read_excel(url, sheet_name="Existing Agent Staffing", engine="openpyxl")

print(df_salary_threshold.head())
print(df_adv_sign_ups.head())
print(df_exist_staff.head())

   Country  Annual_Agent_Salary_USD  Advertiser_Eligibility_Threshold_USD
0      USA                    77721                                 35000
1       UK                    51650                                 27000
2  Germany                    51328                                 26000
3    India                    14118                                 13000
4   Brazil                    27950                                 29000
  Advertiser_ID Country         Sign_Up_Date  Projected_Annual_Budget
0       A000001     USA  2024-01-01 00:00:00                    84376
1       A000002     USA  2024-01-01 00:00:00                    57297
2       A000003     USA  2024-01-01 00:00:00                    34081
3       A000004     USA  2024-01-01 00:00:00                   154506
4       A000005     USA  2024-01-01 00:00:00                    23604
   Country  Existing_Agent_Count
0      USA                   652
1       UK                   589
2  Germany                   602
3   

In [60]:
# Left join the first two tables
eligible_ad = pd.merge(df_adv_sign_ups, df_salary_threshold, on='Country', how='left')
eligible_ad = eligible_ad.drop(columns=['Annual_Agent_Salary_USD'])

# Find eligible advertisers
eligible_ad['Eligible'] = (eligible_ad['Projected_Annual_Budget']> eligible_ad['Advertiser_Eligibility_Threshold_USD'])
eligible_ad = eligible_ad[eligible_ad['Eligible'] == True]

# Drop all on Feb 29, 2025 -> These are the ones that create Error when converting to_datetime
eligible_ad['Sign_Up_Date'] = pd.to_datetime(eligible_ad['Sign_Up_Date'], errors='coerce')
# 2. Drop all rows where 'Sign_Up_Date' became NaT
eligible_ad = eligible_ad.dropna(subset=['Sign_Up_Date'])

eligible_ad.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Advertiser_Eligibility_Threshold_USD,Eligible
0,A000001,USA,2024-01-01,84376,35000,True
1,A000002,USA,2024-01-01,57297,35000,True
3,A000004,USA,2024-01-01,154506,35000,True
7,A000008,USA,2024-01-01,104967,35000,True
8,A000009,USA,2024-01-01,44165,35000,True


In [179]:
# Past Daily Demands
# daily_demand = eligible_ad.groupby(['Country', 'Sign_Up_Date']).size().reset_index(name='eligible_count')

#daily_demand.head()

In [205]:
# Monthly totals
#eligible_ad['Month'] = (eligible_ad['Sign_Up_Date'].dt.to_period('M'))
#monthly_demand = (eligible_ad.groupby(['Country', 'Month']).size())

#monthly_demand.head()

## Three Tools will be built:
### Tool 1. Simulator by country.
<b> This simulates advertiser assignment and agent capacity for a country on a day-by-day bases over a specified simulation period. It follows the following logic: </b>  
- When an eligible advertiser comes in, they sit in a waiting pool.
> - Path 1: If there's an agent available, they are assigned to it. Supported for 60 days, and 'graduate'.  
> - Path 2: No available agent, they will continue waiting. Everyday, if there's an agent available, they will be assigned. After 60 days if they are still unassigned, they are removed from the pool.

<b> Note: the following rules are observed: </b>
> 1. Each agent can support up to 10 advertisers.
> 2. Advertisers are assigned when agent capacity is available, in a first-in, first-out basis.
> 3. Assigned advertisers receive support for 60 days.
> 4. Capacity is released when an advertiser 'graduates'.
> 5. A staffing reduction in the next round cannot reduce the capacity below the number of advertisers already receiving support.

<b> This function simply simulates a scenario based on a staffing plan, it does not decide how many agents to hire or terminate. </b>

In [69]:
def simulate_country(advertisers, country, staffing_plan, start_date="2024-01-01", end_date="2024-12-31"):
    '''
       Inputs: 
             advertisers (df): dataframe containing a list of advertisers by country and sign-up dates. 
                               Required columns: 
                                   Advertiser_ID, Contry, and Sign-Up Dates
                country (str): country to simulate
           staffing_plan (df): dataframe containing staffing plan (agent count) of a country by month
                               Required columns:
                                   Country, Month, Agents
            start_date (date): start date of simulation period
              end_date (date): end date of simulation period

        Output: Two dataframes
            1. advertiser_result:
                One row per advertiser, containing:
                    - advertiser_ID
                    - status: Not arrived (not yet signed-up), Waiting, Assigned, Graduated, Expired
                    - assignment_date, graduation_date, expiration_date
            2. daily_result:
                One row per simulation day, containing:
                    - date, Country, agents, capacity (agent*10)
                    - new_eligible, assigned_today, expired_today, supported, waiting (all counts)
                    - utilization
    '''
    
    df = advertisers[advertisers['Country'] == country].copy()

    df['status'] = 'NOT_ARRIVED'
    df['assignment_date'] = pd.NaT
    df['graduation_date'] = pd.NaT
    df['expiration_date'] = pd.NaT

    daily_results = []

    dates = pd.date_range(start=start_date, end=end_date, freq="D")

    for current_date in dates:

        # Enhancement 1a: dynamic staffing
        current_month = current_date.to_period("M")
        staffing_row = staffing_plan[(staffing_plan["Country"] == country) & (staffing_plan["Month"] == current_month)]
        if staffing_row.empty:
            raise ValueError(f"No staffing data found for " 
                             f"{country}, {current_month}")
        agent_count = int(staffing_row["Agents"].iloc[0])

        # 1. Graduate advertisers
        graduating = ((df["status"] == "ASSIGNED")  & (df["graduation_date"] <= current_date))
        graduated_today = graduating.sum()
        df.loc[graduating, "status"] = "GRADUATED"

        # 2. Add today's advertisers
        arrivals = ((df["Sign_Up_Date"] == current_date) & (df["status"] == "NOT_ARRIVED") )
        new_eligible_today = arrivals.sum()
        df.loc[arrivals, "status"] = "WAITING"

        # 3. Expire old waiting advertisers
        expired = ((df["status"] == "WAITING") & (current_date > df["Sign_Up_Date"]+ pd.Timedelta(days=60)))
        expired_today = expired.sum()
        df.loc[expired, "status"] = "EXPIRED"
        df.loc[expired, "expiration_date"] = current_date

        # 4. Calculate capacity
        total_capacity = agent_count * 10
        currently_supported = (df["status"] == "ASSIGNED").sum()
        available_capacity = max(0, total_capacity - currently_supported)

        # Enhancement 1b: raise error if staff reduction causes currently supported advertisers to suddenly go without an agent
        if currently_supported > total_capacity:
            raise ValueError(f"Infeasible staffing plan: "
                             f"{Country} on {current_date.date()} " f"has {currently_supported} supported advertisers " 
                            f"but only {total_capacity} available capacity.")

        # 5. First-In First-Out assignment
        waiting = (df[df["status"] == "WAITING"].sort_values(["Sign_Up_Date", "Advertiser_ID"]))
        to_assign = waiting.head(available_capacity)
        assigned_today = len(to_assign)
        if assigned_today > 0:
            df.loc[to_assign.index,"status"] = "ASSIGNED"
            df.loc[to_assign.index, "assignment_date"] = current_date
            df.loc[to_assign.index, "graduation_date"] = (current_date + pd.Timedelta(days=60)            )

        # 6. Daily results
        supported = (df["status"] == "ASSIGNED").sum()
        waiting_count = (df["status"] == "WAITING").sum()
        utilization = (supported / total_capacity if total_capacity > 0 else 0)
        daily_results.append({
            "date": current_date,
            "Country": country,
            "agents": agent_count,
            "capacity": total_capacity,
            "new_eligible": new_eligible_today,
            "assigned_today": assigned_today,
            "graduated_today": graduated_today,
            "expired_today": expired_today,
            "supported": supported,
            "waiting": waiting_count,
            "utilization": utilization
        })

    return (df, pd.DataFrame(daily_results))

### Tool 2. Staffing schedule
<b> This converts monthly hiring and termination decisions into the actual productive agent headcount available each month, which calculating total staffing and termination costs. </b>  
<b> Note: the following rules are observed: </b>
> 1. Hiring and termination decisions takes effect after one month.
    - Agents hired in month M will become productive only in month M+1.  
    - Agents to be terminated in month M will leave in month M+1.
> 2. Every month, each agent is paid annual_salary / 12.
> 3. Termination incurs a one-time cost of 0.40 * annual_salary.

<b> This function simply simulates the staffing plan as input for Tool 1. </b>

In [75]:
def build_staffing_schedule(initial_agents, decisions):
    '''
        Inputs:
            - initial_agents (integer): obtained from df_exist_staff
            - decisions (df): Monthly Staffing Decisions
                - Required Columns:
                    - Country
                    - Month ('2026-01')
                    - Hires
                    - Terminations

        Outputs:
            - staffing plan: a dataframe where each row contains:
                - Country
                - Month
                - Agents: available to support advertisers
                - agents_ramping: agents ramping up and not available for support
                - termination_notices: notices issued that month for the country
                - annual_agent_salary_USD, monthly_salary
                - paid_agents, salary_cost, termination_cost, total_cost
    '''
    
    decisions = decisions.sort_values("Month").copy()
    active_agents = initial_agents

    previous_hires = 0
    previous_terminations = 0

    results = []

    for _, row in decisions.iterrows():

        # Decisions from previous month
        # take effect this month
        active_agents += previous_hires
        active_agents -= previous_terminations

        if active_agents < 0:
            raise ValueError(f"Active agents became negative " f"in {row['Month']}")

        results.append({
            "Country": row["Country"],
            "Month": row["Month"],
            "Agents": active_agents,
            "Agents_Ramping": int(row["Hires"]),
            "Hires": int(row["Hires"]),
            "Terminations_Notices": int(
                row["Terminations"]
            )
        })

        previous_hires = int(row["Hires"])
        previous_terminations = int(row["Terminations"])

    staffing_plan = pd.DataFrame(results)
    staffing_plan = staffing_plan.merge(df_salary_threshold[["Country", "Annual_Agent_Salary_USD"]],
                                        on="Country", how="left")
    staffing_plan['Monthly_Salary'] = staffing_plan['Annual_Agent_Salary_USD']/12
    staffing_plan["Paid_Agents"] = (staffing_plan["Agents"] + staffing_plan["Agents_Ramping"])
    staffing_plan["Salary_Cost"] = (staffing_plan["Paid_Agents"] * staffing_plan["Monthly_Salary"])
    staffing_plan["Termination_Cost"] = (staffing_plan["Terminations_Notices"] * staffing_plan["Annual_Agent_Salary_USD"]* 0.40)
    staffing_plan["Total_Cost"] = (staffing_plan["Salary_Cost"] + staffing_plan["Termination_Cost"])

    return staffing_plan

### Tool 3. Evaluate staffling plan
<b> This evaluates the performance of a complete staffing simulation, combining advertiser service outcomes, agent utilization, staffing costs, and expected revenue uplift. </b>  
<b> Note: the following metrics are measured: </b>
> 1. Timely Advertiser Support:
     -  Number of advertisers that received support/expired
     -  Advertiser expiration rate
     -  Average, 90th percentile, and maximum waiting time for agent support
> 2. Cost
     - Total staffing costs across the whole evaluation period
> 3. Revenue Impact
     - Expected uplift E[X] to be calculated:
        - Potential uplift = if every eligible advertiser received support
        - Expected uplift = actual advertisers that received support
> 4. Net financial impact = Revenue - Cost


In [76]:
revenue_uplift = {0.00: 0.05, 0.05: 0.15, 0.10: 0.25, 0.15: 0.25, 0.20: 0.20, 0.25: 0.10}
expected_uplift = sum(uplift * probability for uplift, probability in revenue_uplift.items())

print(expected_uplift)

0.135


In [77]:
def evaluate_plan(advertiser_results, daily_results, staffing_plan):
    '''
        Inputs:
            - advertiser_results: advertiser results from function simulate_country()
            - daily_results: daily results from function simulate_country()
            - staffing_plan: monthly_staffing plan created from function build_staffing_schedule()

        Output:
            - Dictionary containing the following metrics:
                - advertisers supported, advertisers expired, expiration rate, average wait days, percentile 90 waiting days
                - average agent utilization, total staffing cost
                - expected uplift rate, potential revenue uplift, expected revenue uplift
                - net finances: revenue - cost

    '''

    completed = advertiser_results[advertiser_results["assignment_date"].notna()].copy()
    completed["wait_days"] = (completed["assignment_date"] - completed["Sign_Up_Date"]).dt.days
    max_wait = completed["wait_days"].max()
    avg_wait = completed["wait_days"].mean()
    p90_wait = completed["wait_days"].quantile(0.90)
    expired = (advertiser_results["status"] == "EXPIRED").sum()
    expiration_rate = (expired / len(advertiser_results))
    avg_utilization = (daily_results["utilization"].mean())
    peak_utilization = (daily_results["utilization"].max())
    total_staffing_cost = staffing_plan["Total_Cost"].sum()

    expected_uplift_rate = 0.1375
    supported = advertiser_results[advertiser_results["assignment_date"].notna()].copy()
    total_potential_revenue_uplift = (advertiser_results["Projected_Annual_Budget"].sum() * expected_uplift_rate)
    expected_revenue_uplift = (supported["Projected_Annual_Budget"].sum() * expected_uplift_rate)
    net_financial_impact = (expected_revenue_uplift - total_staffing_cost)


    return {
        "advertisers_supported": len(completed),
        "advertisers_expired": int(expired),
        "average_wait_days": avg_wait,
        "p90_wait_days": p90_wait,
        "average_utilization": avg_utilization,
        "expired_advertisers": expired,
        "expiration_rate": expiration_rate,
        "total_staffing_cost": total_staffing_cost,
        "expected_uplift_rate": expected_uplift_rate,
        "potential_revenue_uplift": float(total_potential_revenue_uplift),
        "expected_revenue_uplift": float(expected_revenue_uplift),
        # Overall financial result
        "net_financial_impact":float(net_financial_impact)
    }

### With three functions ready, we can start building for 2026. For this Hackathon, we need to use AI to help with staffing, to understand this, we will compare its results against a baseline.

<b> In the next section, we will build this baseline. To do this: </b>
1. Forecast 2026 demand based on 2024-2025 data.
   - Note that we are making two forecasts: first is the monthly forecast, second is the actual advertiser-level forecast.
2. Using the forecasted information, feed through the three functions.
   -  The baseline scenario will be: 0 hired and 0 termination - use existing agent count.

<b> This baseline result will be used to compare against AI's results. </b>

### Step 1: Find 2026 forecast: both monthly-level and advertiser-level.

In [78]:
# Prepare data for forecast

import numpy as np
import pandas as pd

# Monthly totals
eligible_ad['Month'] = (eligible_ad['Sign_Up_Date'].dt.to_period('M'))
monthly_demand = (eligible_ad.groupby(['Country', 'Month']).size().reset_index(name="Eligible_Advertisers"))
monthly_demand["Year"] = (monthly_demand["Month"].dt.year)
monthly_demand["Month_Number"] = (monthly_demand["Month"].dt.month)

demand_history = (monthly_demand.pivot_table(index=["Country", "Month_Number"],
        columns="Year",
        values="Eligible_Advertisers",
        aggfunc="sum",
        fill_value=0).reset_index())
demand_history = demand_history.rename(columns={2024: "Demand_2024", 2025: "Demand_2025"})

demand_history["Growth_Rate"] = ((demand_history["Demand_2025"] - demand_history["Demand_2024"])/ demand_history["Demand_2024"])
demand_history.head()


Year,Country,Month_Number,Demand_2024,Demand_2025,Growth_Rate
0,Argentina,1,53,90,0.698113
1,Argentina,2,66,70,0.060606
2,Argentina,3,69,59,-0.144928
3,Argentina,4,71,79,0.112676
4,Argentina,5,67,65,-0.029851


<b> Build the first forecast function: this forecasts the monthly eligible advertisers based on 2024-2025 values. </b>

In [80]:
import pandas as pd
import numpy as np

def forecast_2026_monthly_demand(monthly_demand, min_growth=-0.20, max_growth=0.30):
    """
        Forecasts monthly eligible advertiser demand for 2026 using 2024 and 2025 monthly demand.
    
        Forecast logic:
            Growth = (2025 Demand - 2024 Demand) / 2024 Demand
            2026 Forecast = 2025 Demand * (1 + Applied Growth)

        Growth is capped to prevent extreme forecasts:
            min_growth : Minimum allowed growth rate.
                Default = -20%.
            max_growth : Maximum allowed growth rate.
                Default = +30%.

    """

    # Cap extreme growth
    monthly_demand["Applied_Growth"] = (monthly_demand["Growth_Rate"].clip(lower=min_growth, upper=max_growth))

    # Forecast 2026
    monthly_demand["Forecast_2026"] = (monthly_demand["Demand_2025"] * (1 + monthly_demand["Applied_Growth"])).round().astype(int)

    # Create 2026 Month
    monthly_demand["Month"] = pd.PeriodIndex([f"2026-{month:02d}"for month in monthly_demand["Month_Number"]],freq="M")

    # Final output
    forecast = monthly_demand[
        ["Country",
            "Month",
            "Month_Number",
            "Demand_2024",
            "Demand_2025",
            "Growth_Rate",
            "Applied_Growth",
            "Forecast_2026"]].copy()


    return (forecast.sort_values(["Country", "Month"]).reset_index(drop=True))

In [35]:
forecast_2026 = forecast_2026_monthly_demand(demand_history)
forecast_2026.head()

Year,Country,Month,Month_Number,Demand_2024,Demand_2025,Growth_Rate,Applied_Growth,Forecast_2026
0,Argentina,2026-01,1,53,90,0.698113,0.300000,117
1,Argentina,2026-02,2,66,70,0.060606,0.060606,74
2,Argentina,2026-03,3,69,59,-0.144928,-0.144928,50
3,Argentina,2026-04,4,71,79,0.112676,0.112676,88
4,Argentina,2026-05,5,67,65,-0.029851,-0.029851,63


<b> Build the second forecast function: this forecasts eligible advertisers information based on 2026 monthly forecasts, and 2024-2025 values. </b>

In [81]:
def create_2026_forecast_advertisers(forecast_2026, historical_advertisers, random_seed=42):
    """
    Converts monthly 2026 demand forecasts into advertiser-level records for the daily staffing simulator.
    
    historical_advertisers : from 2024-2025
    random_seed : Makes the generated forecast reproducible.

    """

    forecast = forecast_2026.copy()
    historical = historical_advertisers.copy()

    # Ensure dates have the correct type
    historical["Sign_Up_Date"] = pd.to_datetime(historical["Sign_Up_Date"])

    # Ensure only 2024 and 2025 years
    historical = historical[historical["Sign_Up_Date"].dt.year.isin([2024, 2025])].copy()
    historical["Month_Number"] = (historical["Sign_Up_Date"].dt.month)
    historical["Day_Of_Month"] = (historical["Sign_Up_Date"].dt.day)

    rng = np.random.default_rng(random_seed)

    generated_rows = []
    advertiser_number = 1

    # Generate advertisers for each Country + Month
    for _, row in forecast.iterrows():

        country = row["Country"]
        month_number = int(row["Month_Number"])

        forecast_count = int(row["Forecast_2026"])

        # Historical advertisers from the same country and calendar month
        historical_pool = historical[(historical["Country"] == country) & (historical["Month_Number"] == month_number)]

        # Sample historical advertisers
        sampled = historical_pool.sample(n=forecast_count, replace=True, random_state=(random_seed + advertiser_number))
        sampled = sampled.reset_index(drop=True)

        # Create corresponding 2026 records
        for _, hist_row in sampled.iterrows():

            signup_date = pd.Timestamp(year=2026, month=month_number, day=int(hist_row["Day_Of_Month"]))

            generated_rows.append({
                "Advertiser_ID": f"F2026_{advertiser_number:08d}",
                "Country": country,
                "Sign_Up_Date": signup_date,
                "Projected_Annual_Budget": hist_row["Projected_Annual_Budget"],
                "Data_Type":"2026_FORECAST"})

            advertiser_number += 1

    forecast_advertisers = pd.DataFrame(generated_rows)

    return (forecast_advertisers.sort_values(["Country", "Sign_Up_Date"]).reset_index(drop=True))

In [39]:
ad_forecast_2026 = create_2026_forecast_advertisers(forecast_2026, historical_advertisers=eligible_ad)
ad_forecast_2026.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Data_Type
0,F2026_00000036,Argentina,2026-01-02,7026,2026_FORECAST
1,F2026_00000072,Argentina,2026-01-02,8930,2026_FORECAST
2,F2026_00000077,Argentina,2026-01-02,28609,2026_FORECAST
3,F2026_00000103,Argentina,2026-01-02,8930,2026_FORECAST
4,F2026_00000005,Argentina,2026-01-03,9422,2026_FORECAST


### Step 2: Calculate Baseline For All Countries in 2026

In [48]:
all_staffing_plans = []
all_advertiser_results = []
all_daily_results = []
all_baseline_summary = []

for _, row in df_exist_staff.iterrows():

    country = row["Country"]
    agents = int(row["Existing_Agent_Count"])
    baseline_decisions = pd.DataFrame({
        "Country": [country] * 12,
        "Month": pd.period_range(
            "2026-01",
            "2026-12",
            freq="M"),
        "Hires": [0] * 12,
        "Terminations": [0] * 12
    })

    baseline_staffing_plan = build_staffing_schedule(initial_agents=agents, decisions=baseline_decisions)
    advertiser_result, daily_result = simulate_country(ad_forecast_2026, country=country, staffing_plan=baseline_staffing_plan, start_date="2026-01-01", end_date="2026-12-31")

    all_staffing_plans.append(baseline_staffing_plan)
    all_advertiser_results.append(advertiser_result)
    all_daily_results.append(daily_result)

    #evaluate_plan(advertiser_results= advertiser_result,daily_results = daily_result, staffing_plan = baseline_staffing_plan)
    

simulation_daily = pd.concat(all_daily_results, ignore_index=True)
simulation_advertisers = pd.concat(all_advertiser_results,ignore_index=True) 
simulation_staffing = pd.concat(all_staffing_plans, ignore_index=True)

In [82]:
#baseline_staffing_plan.head()
#simulation_daily.head()
#simulation_advertisers.head()
simulation_staffing.head()

,Country,Month,Agents,Agents_Ramping,Hires,Terminations_Notices,Annual_Agent_Salary_USD,Monthly_Salary,Paid_Agents,Salary_Cost,Termination_Cost,Total_Cost
0,USA,2026-01,652,0,0,0,77721,6476.75,652,4222841.0,0.0,4222841.0
1,USA,2026-02,652,0,0,0,77721,6476.75,652,4222841.0,0.0,4222841.0
2,USA,2026-03,652,0,0,0,77721,6476.75,652,4222841.0,0.0,4222841.0
3,USA,2026-04,652,0,0,0,77721,6476.75,652,4222841.0,0.0,4222841.0
4,USA,2026-05,652,0,0,0,77721,6476.75,652,4222841.0,0.0,4222841.0


<b> Since all results are concatenated, the following function wraps it so it's easy to query individual country results. </b>

In [54]:
def evaluate_country(country, simulation_advertisers, simulation_daily, simulation_staffing):

    advertiser_results = simulation_advertisers[simulation_advertisers["Country"] == country]
    daily_results = simulation_daily[simulation_daily["Country"] == country]
    staffing_plan = simulation_staffing[simulation_staffing["Country"] == country]
    metrics = evaluate_plan(advertiser_results=advertiser_results,daily_results=daily_results,staffing_plan=staffing_plan)
    metrics["Country"] = country

    return metrics

In [66]:
argentina_metrics = evaluate_country("USA", simulation_advertisers, simulation_daily, simulation_staffing)

argentina_metrics

{'advertisers_supported': 13046,
 'advertisers_expired': 0,
 'average_wait_days': 0.0,
 'p90_wait_days': 0.0,
 'average_utilization': 0.2976132448104883,
 'expired_advertisers': 0,
 'expiration_rate': 0.0,
 'total_staffing_cost': 50674092.0,
 'expected_uplift_rate': 0.1375,
 'potential_revenue_uplift': 154084310.875,
 'expected_revenue_uplift': 154084310.875,
 'net_financial_impact': 103410218.875,
 'Country': 'USA'}